# Session 14 · Homework — SOLUTIONS (teacher)

Worked solutions with commentary for the teacher. All cells run top to bottom.
**Key teaching point:** students split by hand on *attendance*, but the tree roots on
*study hours* — a deliberate mismatch that previews S15's feature-importance lesson
(the tree tells you which feature it actually found most useful).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

full = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]
sample_ids = ["S169","S303","S221","S343","S352","S024","S347","S312","S172","S062","S257","S237"]
sample = (full[full['student_id'].isin(sample_ids)]
          [['student_id','study_hours_per_week','attendance_pct','passed']]
          .sort_values('attendance_pct').reset_index(drop=True))
sample

## Part 1 · By-hand attendance split — SOLUTION

In [ ]:
def count_split(df, col, thr):
    left, right = df[df[col] < thr], df[df[col] >= thr]
    lp, lf = int(left['passed'].sum()), int((1-left['passed']).sum())
    rp, rf = int(right['passed'].sum()), int((1-right['passed']).sum())
    print(f'{col} < {thr}:  LEFT {len(left)} ({lp}P/{lf}F)   RIGHT {len(right)} ({rp}P/{rf}F)')

def gini(df):
    if len(df)==0: return 0.0
    p = df['passed'].mean(); return 1 - p**2 - (1-p)**2
def wgini(df, col, thr):
    l, r = df[df[col]<thr], df[df[col]>=thr]
    return (len(l)*gini(l)+len(r)*gini(r))/len(df)

for thr in [75, 85]:
    count_split(sample, 'attendance_pct', thr)
    print(f'   weighted impurity: {wgini(sample, "attendance_pct", thr):.3f}')

# TEACHER: the two cuts are close; whichever gives the lower weighted impurity is 'purer'.
# Accept either answer if the student compares the counts. The point is the METHOD:
# split, count pass/fail per side, pick the cleaner pair.

## Part 2 · Depth-1 tree — SOLUTION

The tree roots on **study_hours_per_week <= 7.65**, *not* attendance — even though the
student just split attendance by hand. That's the intended surprise: the tree
measures every feature and reports the most useful one (formally, S15's feature
importance). Test accuracy ≈ **0.865**.

In [ ]:
X, y = full[habits], full['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)
print(export_text(stump, feature_names=habits))
print('test accuracy:', round(stump.score(X_test, y_test), 3))
print()
print('Root feature: study_hours_per_week (NOT attendance).')

## Part 3 · Depth-2 tree, two paths aloud — SOLUTION

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print(export_text(tree2, feature_names=habits))

# Example read-aloud rules (student wording will vary):
print()
print('Path 1: If study hours <= 7.65, predict FAIL.')
print('Path 2: If study hours > 7.65 AND the second question sends you right, predict PASS.')
print('(Exact second-level feature/threshold: read it off export_text above.)')

fig, ax = plt.subplots(figsize=(11,5))
plot_tree(tree2, feature_names=habits, class_names=['fail','pass'], filled=True, ax=ax, fontsize=9)
plt.tight_layout(); plt.show()

## Part 4 · Interpretability — SOLUTION (sample answer)

**The tree splits first on study hours** — the strongest single signal of passing.

A **school counsellor** would prefer this readable tree because they must *explain*
a prediction to a student and parent — "the model flags you because you're studying
under ~8 hours a week" is an actionable, contestable reason. A more accurate but
unreadable model might predict slightly better yet offer no explanation the family
could act on or challenge — and "the computer said so" is not acceptable when a real
person's path is at stake. **Interpretability can be worth a little accuracy** when a
decision must be justified to the people it affects. (This trade-off returns in S18.)

**Acceptable variation:** any well-argued stakeholder (parent, admissions officer,
teacher) and any reasonable by-hand attendance cut. Full marks require comparing the
two counts in Part 1 and naming a real stakeholder in Part 4.